<a href="https://colab.research.google.com/github/aztcc/minerodedatos77/blob/main/Proyecto_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import librosa
import numpy as np

In [2]:
# --- CONFIGURACIÓN GLOBAL DEL PIPELINE ---
SR_TARGET = 22050       # Tasa de muestreo estándar obligatoria (22.05 kHz)
N_MFCC = 13             # Número de coeficientes Mel a extraer por frame
MAX_PAD_LEN = 40        # Longitud horizontal fija (Equivale a ~1.0 segundo de audio con saltos STFT por defecto)

In [3]:
def extract_features_from_file(file_path, n_mfcc=N_MFCC, max_pad_len=MAX_PAD_LEN):
    """
    Carga un archivo de audio, extrae los MFCC y aplica padding o truncamiento
    para devolver una matriz con dimensiones estrictas (n_mfcc, max_pad_len).
    """
    try:
        # 1. Cargar el audio forzando tasa de muestreo objetivo y mono-canal
        audio, sr = librosa.load(file_path, sr=SR_TARGET, mono=True)

        # 2. Extracción de Coeficientes Cepstrales en la Escala Mel
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)

        # 3. Lógica de Homogeneización Temporal (Padding o Truncamiento)
        if mfcc.shape[1] < max_pad_len:
            # Si el audio es corto, se añade relleno con ceros a la derecha (Zero-padding)
            pad_width = max_pad_len - mfcc.shape[1]
            mfcc_padded = np.pad(mfcc, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else:
            # Si el audio excede el tiempo máximo, se truncan los frames adicionales
            mfcc_padded = mfcc[:, :max_pad_len]

        return mfcc_padded
    except Exception as e:
        print(f"⚠️ Error al procesar {file_path}: {e}")
        return None

In [8]:
def load_dataset_pipeline(base_path):
    """
    Escanea las subcarpetas de la ruta base asignando etiquetas binarias.
    Apagar = 0, Encender = 1.
    """
    X_data = []
    y_labels = []

    # Estructura semántica de clases
    categories = {'apagar': 0, 'encender': 1}

    for folder_name, label_value in categories.items():
        folder_path = os.path.join(base_path, folder_name)

        if not os.path.exists(folder_path):
            print(f"❌ Advertencia: No existe la subcarpeta {folder_path}")
            continue

        print(f"📁 Cargando audios de la categoría: [{folder_name}] (Asignando Label: {label_value})...")

        for file_name in os.listdir(folder_path):
            # Filtrar por formatos de audio estándar
            if file_name.lower().endswith(('.wav', '.mp3', '.m4a', '.ogg')):
                full_path = os.path.join(folder_path, file_name)
                features = extract_features_from_file(full_path)

                if features is not None:
                    X_data.append(features)
                    y_labels.append(label_value)

    # Conversión estructural a tensores NumPy nativos
    X_data = np.array(X_data)
    y_labels = np.array(y_labels)

    return X_data, y_labels

In [9]:
# --- Ejecución en Google Colab ---
# 1. Montar drive
# from google.colab import drive
# drive.mount('/content/drive')

# 2. Cargar el dataset (Ruta ejemplo de Drive unificado)
PATH_DATASET = "/content/drive/MyDrive/Dataset_Luz"
X, y = load_dataset_pipeline(PATH_DATASET)

# 3. Normalización MinMax con respecto a los extremos globales del conjunto
X_min = X.min()
X_max = X.max()
X_normalized = (X - X_min) / (X_max - X_min)

print("\n📊 Resumen de la Estructura de Datos:")
print(f"Dimensiones de X (Matriz de Características): {X_normalized.shape}")
# Salida esperada con el total de alumnos: (122, 13, 40)
print(f"Dimensiones de y (Vectores de Etiquetas): {y.shape}")
# Salida esperada: (122,)

❌ Advertencia: No existe la subcarpeta /content/drive/MyDrive/Dataset_Luz/apagar
❌ Advertencia: No existe la subcarpeta /content/drive/MyDrive/Dataset_Luz/encender


ValueError: zero-size array to reduction operation minimum which has no identity

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense

# Inicialización de la Red Secuencial Base
model = Sequential([
    # Recibe la matriz (13, 40) y la aplana a un vector unidimensional de 520 elementos
    Flatten(input_shape=(N_MFCC, MAX_PAD_LEN)),

    # Capas ocultas densas (Aquí inicia la experimentación/tuning de los alumnos)
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),

    # Capa final de clasificación binaria (Salida probabilística entre 0 y 1)
    Dense(1, activation='sigmoid')
])

# Compilación con métricas para análisis de curvas
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 520)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        33,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,457 (138.50 KB)

 Trainable params: 35,457 (138.50 KB)

 Non-trainable params: 0 (0.00 B)